In [38]:
code = """
import os
import numpy as np
import joblib
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing import image
import streamlit as st

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

st.set_page_config(page_title="Fruit Ripeness Detector", layout="centered")

st.title(" Multi-Modal Fruit Ripeness Detector")
st.write("Combine **numeric sensor models** and an **image model** to predict fruit ripeness or spoilage.")

# Feature definitions
gas_env_features = [
    'Temp-int','Press-int','Humid-int','Temp-ext','Press-ext','Humid-ext',
    'TGS20','TGS02','SGP'
]
specter_features = [
    'SpA410','SpB435','SpC460','SpD485','SpE510','SpF535','SpG560','SpH585',
    'SpR610','SpI645','SpS680','SpJ705','SpT730','SpU760','SpV810','SpW860',
    'SpK900','SpL940'
]
all_features = gas_env_features + specter_features

label_map = {
    0: "Underripe",
    1: "Ripe",
    2: "Overripe",
    3: "Rotten",
    4: "Severely Rotten"
}

# ------------------------------------------------------------
# MODEL CLASSES
# ------------------------------------------------------------

class SimpleModel:
    def __init__(self, name, features, model_path=None):
        self.name = name
        self.features = features
        self.model_path = model_path
        self.scaler = None

        try:
            if model_path and os.path.exists(model_path):
                self.model = joblib.load(model_path)
                st.success(f"Loaded model: {name}")
            else:
                raise FileNotFoundError("Model path not found")
        except Exception as e:
            st.warning(f"Using dummy model for {name} ({e})")
            self.scaler = StandardScaler()
            X_dummy = np.random.rand(50, len(self.features))
            y_dummy = np.random.randint(0, 5, 50)
            X_dummy = self.scaler.fit_transform(X_dummy)
            self.model = RandomForestClassifier(random_state=42)
            self.model.fit(X_dummy, y_dummy)

    def predict(self, values):
        X = np.array(values).reshape(1, -1)
        if self.scaler is not None:
            X = self.scaler.transform(X)
        return int(self.model.predict(X)[0])


# ------------------------------------------------------------
# LOAD MODELS
# ------------------------------------------------------------

models = {
    "Gas & Env": SimpleModel("Gas & Env", gas_env_features, "/content/random_forest_model_gas&env.pkl"),
    "All": SimpleModel("All", all_features, "/content/random_forest_model_all.pkl"),
    "Specter": SimpleModel("Specter", specter_features, "/content/random_forest_model_specter.pkl")
}

image_model = None
if os.path.exists("image_classification_model.h5"):
    image_model = tf.keras.models.load_model("/content/image_classification_model.h5")
    st.success("Loaded image model.")
else:
    st.warning("Image model not found (image_classification_model.h5).")


# ------------------------------------------------------------
# STREAMLIT UI
# ------------------------------------------------------------

st.header("Select Models to Use")
selected_models = st.multiselect(
    "Choose one or more models:",
    ["Gas & Env", "All", "Specter", "Image Model"]
)

results = {}

# -------------------- Numeric Inputs ------------------------

for model_name in selected_models:
    if model_name == "Image Model":
        continue  # handled below

    model = models.get(model_name)
    if not model:
        continue

    st.subheader(f"Enter inputs for {model_name}")
    cols = st.columns(3)
    values = []

    for i, feature in enumerate(model.features):
        with cols[i % 3]:
            val = st.number_input(feature, value=0.0, key=f"{model_name}_{feature}")
            values.append(val)

    if st.button(f"Predict ({model_name})"):
        pred = model.predict(values)
        results[model_name] = pred
        st.success(f"{model_name} → Class {pred} ({label_map[pred]})")

# -------------------- Image Input ---------------------------

if "Image Model" in selected_models:
    st.subheader("Upload an image for prediction")
    img_file = st.file_uploader("Upload image", type=["jpg", "jpeg", "png"])

    if img_file is not None and image_model is not None:
        img = image.load_img(img_file, target_size=image_model.input_shape[1:3])
        st.image(img, caption="Uploaded Image", use_column_width=True)

        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)


        pred = image_model.predict(img_array)
        pred_class = int(np.argmax(pred, axis=1)[0])

        # Map image model (0–3) → numeric (0–4)
        image_to_numeric_map = {0: 4, 1: 1, 2: 2, 3: 0}
        mapped_pred = image_to_numeric_map.get(pred_class, pred_class)

        st.success(f"Image model raw: {pred_class}, mapped to numeric: {mapped_pred} ({label_map[mapped_pred]})")
        results["Image"] = mapped_pred

# -------------------- Fusion ---------------------------

if len(results) > 0 and st.button("Run Final Fusion"):
    st.header("Results")

    for name, pred in results.items():
        st.write(f"{name}: Class {pred} ({label_map[pred]})")

    # Combine results
    final = int(round(np.mean(list(results.values()))))
    st.success(f"Final (Late Fusion) Result: Class {final} ({label_map[final]})")

else:
    st.caption("Select models, enter data or upload an image, then run fusion.")

"""

with open("app_multimodal_streamlit.py", "w") as f:
    f.write(code)


In [39]:
from pyngrok import ngrok
import time, threading, os

# (Paste your token below)
ngrok.set_auth_token("35IhMtsUHRJSTSOlFrLMSxVGNOG_4vGWcHsUeck751aRBGXDb")

# Start Streamlit in a background thread
def run_streamlit():
    os.system("streamlit run app_multimodal_streamlit.py --server.port 8501 --server.address 0.0.0.0")

thread = threading.Thread(target=run_streamlit)
thread.start()

# Wait a few seconds for Streamlit to start
time.sleep(5)

# Create a public tunnel
public_url = ngrok.connect(8501)
print("Your Streamlit app is live at:", public_url)


Your Streamlit app is live at: NgrokTunnel: "https://eunice-feastless-biserially.ngrok-free.dev" -> "http://localhost:8501"


In [36]:
ngrok.kill()